# 05 · Evaluation and calibration

Run each cell with **Shift+Enter**. Cells under **Your turn** contain `...` placeholders; replace them, run, then run the check beneath. Checks print `PASS` or `NOT YET` and never stop the notebook. Worked answers are in the matching `_solutions` notebook.

The checks compare your results with the **truth in `python/simulate.py`**, so they keep working if you change the simulator's settings.

AUC says whether a model **ranks** members correctly. A lapse forecast also needs the **level** right: if it says 2%, about 2% should lapse. That is calibration, and it is what an actuary checks first.

In [ ]:
def check(label, get, expected=True):
    """Print PASS or NOT YET. `get` is a lambda, so missing names or errors never stop the notebook."""
    try:
        got = get()
    except Exception as e:
        print(f"NOT YET  {label}: {type(e).__name__}: {e}")
        return
    try:
        ok = got.equals(expected) if hasattr(got, "equals") else bool(got == expected)
        if isinstance(expected, float) and isinstance(got, (int, float)):
            ok = abs(got - expected) < 1e-6
    except Exception:
        ok = False
    print(f"PASS     {label}" if ok else f"NOT YET  {label}: expected {expected!r}, got {got!r}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

preds = pd.read_parquet("../data/test_predictions.parquet")
preds.head()

## 1. Three scores, three questions

| Metric | Question it answers | Better |
| --- | --- | --- |
| AUC | Does it rank lapsers above stayers? | higher |
| Log loss | How surprised is it by what happened? | lower |
| Brier score | Mean squared error of the probabilities | lower |

A useful floor: a "model" that predicts the training lapse rate for everyone (`train_rate`).

### Your turn 1.1

Build `scores`: a DataFrame with one row per model (`baseline`, `logistic`, `boosting`) and columns `auc`, `log_loss` and `brier`. For the baseline, use `preds["train_rate"]` as the prediction; its AUC is 0.5 by definition.

In [ ]:
scores = ...

In [ ]:
check("1.1 logistic beats the baseline on log loss", lambda: scores.loc["logistic", "log_loss"] < scores.loc["baseline", "log_loss"])
check("1.1 boosting beats the baseline on Brier", lambda: scores.loc["boosting", "brier"] < scores.loc["baseline", "brier"])

## 2. Calibration

`calibration_curve` bins members by predicted probability and compares the average prediction in each bin with the actual lapse rate. A calibrated model sits on the diagonal. Use `strategy="quantile"` so each bin holds the same number of rows; lapse probabilities are small and clustered.

### Your turn 2.1

Plot calibration curves for both models on one chart (10 quantile bins), with a dashed diagonal for reference.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
...
plt.show()

### Your turn 2.2: actual vs expected

Set `ae`: a dict of model name to actual lapses ÷ the sum of predicted probabilities on the test set. Then build `ae_by_tier`: the same ratio for the logistic model, by `product_tier`.

In [ ]:
ae = ...
ae_by_tier = ...

In [ ]:
check("2.2 logistic A/E within 15%", lambda: 0.85 < ae["logistic"] < 1.15)
check("2.2 every tier within 20%", lambda: ae_by_tier.between(0.8, 1.2).all())

## 3. If calibration is off

A model can rank well but sit too high or too low, for example when lapse rates drift over time. The standard fixes, in order of preference:

1. **Fix the cause:** a missing feature, such as tenure or season, or a training window that no longer reflects the present
2. **Recalibrate:** fit `sklearn.calibration.CalibratedClassifierCV` (isotonic or sigmoid) on a recent validation period that the model did not train on
3. **Scale:** multiply predictions by a recent A/E ratio, the actuarial "experience adjustment"

Note in your README whether your models needed any of these, and why.

## Wrap-up

Commit from VS Code: `Evaluation: scores, calibration and A/E`.